In [1]:
pip install transformers datasets torch pandas scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [35]:
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    pipeline
)
from pathlib import Path
import numpy as np
import pandas as pd
import sqlite3
from collections import Counter

In [3]:
main_dir = Path.cwd()

In [13]:
DB_PATH = 'Database/game_recommendation_system.db'
GAME_TEXT_DATA_PATH = 'text_data'

In [34]:
connection = sqlite3.connect(DB_PATH)
connection.row_factory = sqlite3.Row
cursor = connection.cursor()

model_best = AutoModelForSequenceClassification.from_pretrained(
    main_dir / "BERT-GameTox" / "checkpoint-3500"
)
tokenizer_best = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

classifier = pipeline("text-classification", model=model_best, tokenizer=tokenizer_best)

toxicity_label_column_dict = {
    1 : 'insults_flaming',
    2 : 'offensive_texts',
    3 : 'hate_harassment'
}

cursor.execute('SELECT id, title from video_games')

for row in cursor.fetchall():

    game_title = "".join(c for c in row['title'] if c.isalnum() or c in (' ', '-', '_')).strip().replace(' ', '_')
    df_game = pd.read_csv(main_dir / GAME_TEXT_DATA_PATH / f"{game_title}.csv")

    preds = classifier(df_game['text'].tolist(), truncation=True, max_length=512)
    preds = [(model_best.config.label2id[pred['label']]) for pred in preds]

    label_counts = Counter(preds)
    total = len(preds)

    print(f"Label Prevalence for Video Game {row['title']}:")
    for label_id, count in sorted(label_counts.items()):
        if label_id != 0:
            ratio = np.round((count / total), 4)
            print(f"Label {toxicity_label_column_dict[label_id]}: {ratio}")
            cursor.execute(f'''
            UPDATE game_toxicity_prevalence_profile SET {toxicity_label_column_dict[label_id]}=? 
            WHERE game_id=?
            ''', (ratio, row['id']))
        else:
            ratio = np.round((count / total), 4)
            print(f"Label non-toxic: {ratio}")

connection.commit()
connection.close()
    

    

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Label Prevalence for Video Game Counter-Strike 2:
Label non-toxic: 0.7016
Label insults_flaming: 0.1653
Label offensive_texts: 0.1246
Label hate_harassment: 0.0085
Label Prevalence for Video Game Helldivers 2:
Label non-toxic: 0.8522
Label insults_flaming: 0.0723
Label offensive_texts: 0.0723
Label hate_harassment: 0.0033
Label Prevalence for Video Game Dota 2:
Label non-toxic: 0.7362
Label insults_flaming: 0.1079
Label offensive_texts: 0.153
Label hate_harassment: 0.0028
Label Prevalence for Video Game Team Fortress 2:
Label non-toxic: 0.7885
Label insults_flaming: 0.1205
Label offensive_texts: 0.0623
Label hate_harassment: 0.0287
Label Prevalence for Video Game Terraria:
Label non-toxic: 0.9524
Label insults_flaming: 0.0244
Label offensive_texts: 0.0219
Label hate_harassment: 0.0013
Label Prevalence for Video Game Tom Clancys Rainbow Six Siege:
Label non-toxic: 0.8059
Label insults_flaming: 0.0941
Label offensive_texts: 0.0956
Label hate_harassment: 0.0044
Label Prevalence for Video 